[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/06_linear_regression/05_polynomial_regression.ipynb)

# Bài 05 — Hồi quy đa thức (Polynomial Regression)

## 1. Polynomial Regression

Polynomial Regression (hồi quy đa thức) là dạng hồi quy trong đó $y$ phụ thuộc vào $x$ theo một **đa thức** bậc $d$. Ví dụ hàm giả thuyết: $h_\theta(x) = \theta_0 + \theta_1 x + \theta_2 x^2$, hay tổng quát hơn với các hàm cơ sở khác: $h_\theta(x) = \theta_0 + \theta_1 x + \theta_2\sqrt{x}$...

<img src="https://raw.githubusercontent.com/pickus91/Polynomial-Regression-From-Scratch/master/Figures/hypothesis.PNG" style="width:35%;height:35%;">

**Điểm mấu chốt:** model vẫn **tuyến tính theo tham số $\theta$** — ta chỉ tạo thêm các **feature mới** $x^2, x^3, \dots$ rồi chạy Linear Regression trên các feature đó. Mọi lý thuyết của hồi quy tuyến tính (cost function, gradient descent, normal equation) đều áp dụng nguyên vẹn; chỉ khác là đồ thị dự đoán theo $x$ là đường cong.

<img src="images/image-7.gif" style="width:35%;height:35%;">

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/06_linear_regression

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

from ml_course.visualization import set_style

set_style()

## 2. Code Demo — lương theo cấp bậc

### 2.1 Dữ liệu

Bộ `Position_Salaries.csv`: 10 cấp bậc vị trí (Business Analyst → CEO) và mức lương tương ứng.

In [ ]:
dataset = pd.read_csv("../../data/sample/Position_Salaries.csv")
X = dataset[["Level"]].to_numpy(dtype=float)        # ma trận (10, 1)
y = dataset["Salary"].to_numpy(dtype=float)
dataset

### 2.2 Trực quan hoá dữ liệu

In [ ]:
plt.scatter(X, y, color="red")
plt.title("Position Level vs Salary")
plt.xlabel("Position level")
plt.ylabel("Salary")
plt.show()

Lương tăng **theo cấp số nhân** ở các cấp cao — đường thẳng không thể khớp tốt.

### 2.3 Sử dụng scikit-learn

`PolynomialFeatures(degree=4)` biến $x$ thành $[1, x, x^2, x^3, x^4]$. *(bổ sung)* Thêm `StandardScaler` vì $x^4$ có thang đo gấp hàng nghìn lần $x$ — không ảnh hưởng nghiệm của `LinearRegression` nhưng cần thiết nếu dùng gradient descent hay regularization.

In [ ]:
lin_reg = LinearRegression().fit(X, y)
poly_model = make_pipeline(PolynomialFeatures(degree=4), StandardScaler(), LinearRegression()).fit(X, y)
print(PolynomialFeatures(degree=4).fit_transform([[3.0]]))     # minh hoạ: x = 3 → [1, 3, 9, 27, 81]

X_grid = np.arange(X.min(), X.max() + 0.01, 0.1).reshape(-1, 1)   # lưới mịn để đường cong mượt
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, (name, model) in zip(axes, [("Linear (bậc 1)", lin_reg), ("Polynomial bậc 4", poly_model)]):
    ax.scatter(X, y, color="red")
    ax.plot(X_grid, model.predict(X_grid), color="blue")
    ax.set_title(f"{name}: R² trên dữ liệu train = {model.score(X, y):.3f}")
    ax.set_xlabel("Position level")
    ax.set_ylabel("Salary")
plt.show()

### 2.4 Dự đoán kết quả

Một ứng viên có cấp bậc **6.5** (giữa Region Manager và Partner) — mức lương hợp lý là bao nhiêu?

In [ ]:
print("Linear      :", round(lin_reg.predict([[6.5]])[0]))
print("Polynomial 4:", round(poly_model.predict([[6.5]])[0]))

> 📝 *Sửa so với bản gốc:* bản gốc gọi thừa `poly_reg.fit(X_poly, y)` (PolynomialFeatures không có tham số học từ y) và gọi `fit_transform` khi dự đoán; dùng `Pipeline` gọn và tránh nhầm lẫn này.

## 3. Cẩn thận: chỉ 10 điểm dữ liệu! *(bổ sung)*

R² trên **chính dữ liệu train** luôn tăng khi tăng bậc — với bậc 9, đa thức đi **qua đúng** 10 điểm (R² = 1). Nhưng hãy nhìn nó giữa các điểm và khi **ngoại suy**:

In [ ]:
X_wide = np.linspace(0.5, 11, 300).reshape(-1, 1)
fig, axes = plt.subplots(1, 3, figsize=(18, 4.2))
for ax, d in zip(axes, [2, 4, 9]):
    model = make_pipeline(PolynomialFeatures(d), StandardScaler(), LinearRegression()).fit(X, y)
    ax.scatter(X, y, color="red", zorder=3)
    ax.plot(X_wide, model.predict(X_wide), color="blue")
    ax.set_ylim(-3e5, 1.6e6)
    ax.axvspan(10, 11, color="gray", alpha=0.2, label="ngoại suy")
    ax.set_title(f"bậc {d}: R² train = {model.score(X, y):.4f}")
    ax.legend(loc="upper left")
plt.show()

- Bậc 9 khớp hoàn hảo dữ liệu train nhưng **dao động vô lý** giữa các điểm và cực kỳ tệ khi ngoại suy (level 11) → **overfitting** (chương 05, bài 04).
- Đa thức bậc cao **không ngoại suy được** một cách đáng tin cậy — giá trị bùng nổ khi đi ra ngoài vùng dữ liệu.
- Với dữ liệu tăng theo cấp số nhân như lương, một lựa chọn tốt hơn thường là **log-transform target**: fit $\log(\text{salary})$ theo level bằng đường thẳng.

In [ ]:
log_model = LinearRegression().fit(X, np.log(y))
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(X, y, color="red", zorder=3)
ax.plot(X_wide, np.exp(log_model.predict(X_wide)), color="green", label="log-linear: y = exp(a + b·level)")
ax.legend()
ax.set_title("Log-transform target: 2 tham số, đường cong hợp lý, ngoại suy ổn định hơn")
plt.show()
print("Dự đoán level 6.5 (log-linear):", round(np.exp(log_model.predict([[6.5]]))[0]))

## 4. Chọn bậc đa thức bằng validation *(bổ sung)*

Với dữ liệu đủ lớn, chọn bậc bằng cross-validation thay vì nhìn R² train:

In [ ]:
from sklearn.model_selection import cross_val_score

from ml_course.datasets import make_regression_1d

Xs, ys = make_regression_1d(n=120, kind="poly", noise=1.5, seed=0)       # hàm thật là đa thức bậc 3
degrees = range(1, 13)
cv_mse = [-cross_val_score(make_pipeline(PolynomialFeatures(d), StandardScaler(), LinearRegression()),
                           Xs, ys, cv=5, scoring="neg_mean_squared_error").mean() for d in degrees]
best = degrees[int(np.argmin(cv_mse))]
plt.plot(degrees, cv_mse, "o-")
plt.axvline(best, color="gray", ls="--", label=f"bậc tốt nhất = {best}")
plt.yscale("log")
plt.xlabel("bậc đa thức")
plt.ylabel("MSE (5-fold CV)")
plt.legend()
plt.show()

CV chọn đúng bậc gần 3 — bậc của hàm sinh dữ liệu. Ở bài 07, ta sẽ thấy cách khác: dùng bậc cao nhưng **regularization** để kiểm soát độ phức tạp.

## 5. Nhiều biến: số feature bùng nổ *(bổ sung)*

Với $d$ feature và bậc $p$, `PolynomialFeatures` tạo $\binom{d + p}{p}$ cột (gồm cả tương tác $x_1x_2$...):

In [ ]:
from math import comb

for d in [2, 10, 50]:
    print(f"d = {d:3d} feature: bậc 2 → {comb(d + 2, 2):6d} cột | bậc 3 → {comb(d + 3, 3):8d} cột")

→ Đa thức bậc cao chỉ phù hợp với **ít** feature. Với nhiều feature, dùng model phi tuyến khác (tree, kernel, neural network) hoặc chọn lọc vài tương tác có ý nghĩa (chương 17).

## 🧠 Kiểm tra nhanh

1. Vì sao Polynomial Regression vẫn được gọi là "linear" regression?
2. Với 10 điểm dữ liệu, đa thức bậc mấy đi qua đúng mọi điểm? Có nên dùng không?
3. Vì sao cần scale feature khi dùng `PolynomialFeatures` cùng gradient descent hay regularization?

## Tài liệu tham khảo

[1] [CS229 — Machine Learning Course](http://cs229.stanford.edu) · [2] scikit-learn — [Polynomial and Spline interpolation](https://scikit-learn.org/stable/auto_examples/linear_model/plot_polynomial_interpolation.html)

---
<sub>Bài gốc: *Polynomial Regression* — [AIVN-Machine-Learning](https://github.com/thanhhff/AIVN-Machine-Learning), Tuần 3 (MIT License). Đã sửa: dùng Pipeline thay vì fit thừa; bổ sung: overfitting & ngoại suy với 10 điểm, log-transform target, chọn bậc bằng CV, bùng nổ số feature.</sub>